In [2]:
## Read the shakspare file 
## read it in to inspect it 

with open('input.txt','r',encoding = 'utf-8') as f :
    text = f.read()

In [3]:
print("The length of the datasets in characters is : ", len(text))

The length of the datasets in characters is :  1115394


In [4]:
print(text[:100]) # first 100 characters of tiny datasets shakspare 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
# Here all the unique character that occurs in that occur   in this text 
chars = sorted(list(set(text))) # text is a sequence of characters in python 
vocab_size = len(chars) # vocab_size are the possible elements of the sequences 
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### # Create character-to-integer mapping (tokenizer)

In [6]:
# Create character-to-integer mapping (tokenizer)
stoi = {ch: i for i, ch in enumerate(chars)} # string to int
itos = {i: ch for i, ch in enumerate(chars)} # int to string
encode = lambda s: [stoi[c] for c in s] # "hello" → [46, 43, 50, 50, 53]
decode = lambda l: ''.join([itos[i] for i in l]) # [46, 43, ...] → "hello"


# Test it
print(encode("hello"))
print(decode(encode("hello"))) # should print "hello"
print(decode([46, 43, 50, 50, 53]))

## Currently we are only using character level tokenizer 

[46, 43, 50, 50, 53]
hello
hello


In [7]:
# Lets now encode the entire text database and store   it into a torch.tensor

## # Convert entire text to tensor of integers

import torch 
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape , data.dtype)

print(data[:1000]) 

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

* now here it converts all the character sequence into integer . 


In [8]:
## Now seperating the data into train / test(validation).
n = int(0.9*len(data)) # 90 % training data set and rest 10% will be test data set.
train_data = data[:n]
val_data = data[n:] # test data set ( 10% of total data )


* we never going to actually feed the entire data(TEXT) all at once . becoz that will be comutansouly very expensive .


* we only works with chunks of datasets , and when we train the transformer we basically sample random level of sets out ot training set and train just that chunks at a time . 

* and a maximum length that we use for different chunks , let say it is block_size 

In [9]:
block_size = 8 # how many characters the model looks at to predict next
print(train_data[:block_size])

# A sample chunk of training data
x = train_data[:block_size] # input 
y = train_data[1:block_size+1] # ouput ( Traget shifted by 1 )

for t in range(block_size):
    context = x[:t+1]
    target = y[t]

    print(f"When input is {context} then output is {target}")

    


tensor([18, 47, 56, 57, 58,  1, 15, 47])
When input is tensor([18]) then output is 47
When input is tensor([18, 47]) then output is 56
When input is tensor([18, 47, 56]) then output is 57
When input is tensor([18, 47, 56, 57]) then output is 58
When input is tensor([18, 47, 56, 57, 58]) then output is 1
When input is tensor([18, 47, 56, 57, 58,  1]) then output is 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) then output is 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) then output is 58


##### why target shifted by 1 . 
* Language modeling is strictly next-token prediction. Every token in x at step t has its ground-truth next token waiting at t+1.

* here each number repesenting a character or symbol or space out of those 65. 

#### This section implements the data pipeline that feeds batches of sequences into the model during training and validation

In [10]:
torch.manual_seed(19)
batch_size = 4 # how many independent sequence will be process parallel ?
block_size = 8 # what is the maximum context length for predictions ?

def get_batch(split):

    # generate a small batch of data of inputs x and target y 
    data = train_data if split == 'train' else val_data 
    ix = torch.randint(len(data) - block_size , (batch_size,)) # ix will be a 4 number that will be randomly generated b/w 0 and (len(data) - block_size)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb, yb = get_batch('train')
print('input')
print(xb.shape)
print(xb)

print('Target')
print(yb.shape)
print(yb)

print('-------------------------------------------')

for b in range(batch_size): # batch dimension 
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]

        print(f"when input is {context.tolist()} the target is :{target} ")



input
torch.Size([4, 8])
tensor([[46, 47, 51, 12,  0,  0, 15, 20],
        [58, 53,  1, 59, 57, 43,  1, 47],
        [46,  1, 63, 53, 59,  1, 61, 43],
        [42, 39, 63, 57,  5,  1, 56, 43]])
Target
torch.Size([4, 8])
tensor([[47, 51, 12,  0,  0, 15, 20, 30],
        [53,  1, 59, 57, 43,  1, 47, 58],
        [ 1, 63, 53, 59,  1, 61, 43, 56],
        [39, 63, 57,  5,  1, 56, 43, 57]])
-------------------------------------------
when input is [46] the target is :47 
when input is [46, 47] the target is :51 
when input is [46, 47, 51] the target is :12 
when input is [46, 47, 51, 12] the target is :0 
when input is [46, 47, 51, 12, 0] the target is :0 
when input is [46, 47, 51, 12, 0, 0] the target is :15 
when input is [46, 47, 51, 12, 0, 0, 15] the target is :20 
when input is [46, 47, 51, 12, 0, 0, 15, 20] the target is :30 
when input is [58] the target is :53 
when input is [58, 53] the target is :1 
when input is [58, 53, 1] the target is :59 
when input is [58, 53, 1, 59] the ta

##### The Transformer processes all examples in \(X\) simultaneously and predicts the next token at every position. Its predictions are compared with the corresponding targets in \(Y\) to calculate the loss, which helps the model learn.

In [11]:
print(xb) # batch of input that we would like to feed it into that transformer . 

tensor([[46, 47, 51, 12,  0,  0, 15, 20],
        [58, 53,  1, 59, 57, 43,  1, 47],
        [46,  1, 63, 53, 59,  1, 61, 43],
        [42, 39, 63, 57,  5,  1, 56, 43]])


#### What Is a Bigram Model?

A **Bigram Model** predicts the next token \(x_{t+1}\) using only the current token \(x_t\). It ignores all earlier tokens.

$$
P(x_{t+1}\mid x_1,x_2,\ldots,x_t)
\approx P(x_{t+1}\mid x_t)
$$

**Example:** In the sentence "I love machine learning," a bigram model predicts the next word using only the immediately preceding word, not the entire sentence.

**Remember:** Bigram = current token → next token.

In [12]:
# Lets start this feediing into neural networks 

## we are going to start with simplest possible neural network , in case of language modeling is "Bigram Language model"

import torch 
import torch.nn as nn 
from torch.nn import functional as F 

torch.manual_seed(19)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self,idx,targets):


        logits = self.token_embedding_table(idx) # (B,T,C)
        B, T, C = logits.shape
        logits = logits.view(B*T,C)
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)

        return logits, loss

m = BigramLanguageModel(vocab_size)


logits, loss = m(xb, yb)
print(logits.shape)
print(loss)



torch.Size([32, 65])
tensor(4.3980, grad_fn=<NllLossBackward0>)


##### Now we can have evaluate the quality of the model as we have loss ~ 4.3980 and loss = -log(1/65) ~ 4.17438 which as very close entropy 

In [13]:
#### as we have done evaluation , now we want to generate to the model also . 

import torch 
import torch.nn as nn 
from torch.nn import functional as F 

torch.manual_seed(19)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self,idx,targets = None):


        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets is None : 
            loss = None 
        else :     
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

       #### GENARATE FUNCTION FOR THE MODEL ####
## The generate method implements the autoregressive sampling loop that allows the model to produce continuous text token by token:

    #  The job of Generate fn.  is to extent (B, T) --> (B, T+1) --> (B, T+2) ...... continous in all the batch dimensions and time dimensions as many as max_new_tokens 
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context 
        # max_new_tokens : An integer dictating how many new characters to generate iteratively.




        for _ in range(max_new_tokens):
            #get predictions 
            logits, loss = self(idx)

            #focus only on the last time step 
            logits = logits[:, -1, :] # become (B, C)

            # apply softmax to get probablities 
            probs = F.softmax(logits, dim=1) # (B, C) # Converting logits to probablities 

            # sample from the distribution (Stochastic Sampling (torch.multinomial(probs, num_samples=1))):
            idx_next = torch.multinomial(probs, num_samples=1) #(B, 1)
# Rather than greedily picking torch.argmax(probs) (which often produces repetitive loops), torch.multinomial samples token indices based on their assigned probabilities.

            # append sample index to the running sequences 
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
        


m = BigramLanguageModel(vocab_size)


logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1),dtype = torch.long)
generated_ids = m.generate(idx, max_new_tokens=100)[0].tolist()
print(decode(generated_ids))


torch.Size([32, 65])
tensor(4.3980, grad_fn=<NllLossBackward0>)

iyd$zGaBc,e!lc?$uXTNfiatW.sSD'hg!LthQN,c'dq? I,DHRqZFaFwZcuX?QU,CNwDrOVr'hlfgOPDEsw-p3hJZz-uXhJwEdmd


# Bigram Language Model — Complete Code Explanation

## 1. Imports and Random Seed

- `torch`: Performs tensor operations.
- `torch.nn`: Provides neural-network components.
- `F`: Provides functions such as `softmax()` and `cross_entropy()`.
- `torch.manual_seed(19)`: Helps make random initialization reproducible.

## 2. Model Initialization

`nn.Module` is the base class for PyTorch models. `super().__init__()` initializes the parent class.

Your embedding table has shape **(65, 65)** because `vocab_size = 65`.

- 65 rows represent current characters.
- 65 columns represent possible next characters.
- Total learnable parameters: \(65 \times 65 = 4225\).

**Example:** When the current character is `a`, the model retrieves its row containing 65 scores for the possible next characters.

## 3. Forward Method

The `forward()` method generates predictions and optionally calculates loss.

- `idx`: Input character IDs.
- `targets`: Correct next-character IDs.
- `logits`: Scores for all 65 possible next characters.

**Example:**

| Input | Correct next character |
|---|---|
| `a` | `b` |
| `b` | `c` |
| `c` | `d` |

A bigram model predicts the next character using only the current character.

## 4. Handling Missing Targets

When `targets=None`, the model generates logits but does not calculate loss. This is useful during text generation.

When targets are provided, the model compares predictions against the correct answers.

## 5. Reshaping the Tensors

The logits initially have shape `(B, T, C)`:

- `B`: Batch size.
- `T`: Sequence length.
- `C`: Number of possible characters, 65.

The code combines the batch and sequence dimensions to calculate loss.

**Example:**

- Logits: `(4, 8, 65)` → `(32, 65)`
- Targets: `(4, 8)` → `(32,)`

There are 32 predictions, each with 65 possible classes. Reshaping changes the arrangement, not the actual predictions.

## 6. Cross-Entropy Loss

Cross-entropy measures how well the model predicts the correct next character.

\[
L=-\log(P(\text{correct character}))
\]

**Example:**

- Correct character probability = 0.05 → loss ≈ 2.996.
- Correct character probability = 0.80 → loss ≈ 0.223.

Higher probability for the correct character means lower loss.

Your initial loss is **4.3980**. The uniform-prediction baseline for 65 characters is:

\[
\log(65)\approx4.174
\]

Your loss is in the same general range, which is consistent with randomly initialized weights.

## 7. Starting Text Generation

The initial tensor has shape `(1, 1)` and contains token ID `0`.

This represents the first character in the generated sequence. It is not necessarily a special start token.

## 8. Generating the Next Character

The generation loop repeats these steps:

1. **Get logits:** Predict scores for possible next characters.
2. **Select the last position:** Use only the last character's scores.
3. **Apply softmax:** Convert the scores into probabilities.
4. **Sample a token:** `torch.multinomial()` chooses a character according to those probabilities.
5. **Append the token:** `torch.cat()` adds it to the existing sequence.

**Example:**

Before sampling: `[0, 17, 9]`

Sampled token: `[24]`

After appending: `[0, 17, 9, 24]`

This process repeats 100 times when `max_new_tokens=100`.

## 9. Decoding the Generated Text

The generated tensor contains the starting token plus 100 new tokens, giving it shape `(1, 101)`.

- `[0]` selects the first sequence.
- `.tolist()` converts its token IDs into a Python list.
- `decode()` converts those IDs into characters.

Your output looks like gibberish because the model has not learned character patterns yet.

## 10. The Missing Training Workflow

Training teaches the model which characters are likely to follow other characters by updating the embedding table's weights.

### Step 1: Create an optimizer

```python
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
```

- `AdamW`: An optimizer that updates model parameters using gradients.
- `m.parameters()`: The learnable embedding weights.
- `lr=1e-3`: Learning rate of 0.001, controlling the size of parameter updates.

### Step 2: Repeat the training process

```python
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
```

**What happens in each step?**

1. `get_batch('train')`: Samples training inputs (`xb`) and their correct next-character targets (`yb`).
2. `m(xb, yb)`: Predicts next-character scores and calculates loss.
3. `optimizer.zero_grad(set_to_none=True)`: Clears gradients left over from the previous step.
4. `loss.backward()`: Calculates gradients showing how the loss changes with the model's parameters.
5. `optimizer.step()`: Updates the embedding weights to help reduce the loss.

The model repeats this process 10,000 times. The exact number of steps and learning rate may need adjustment.

### Step 3: Generate text after training

Once training is complete, use the same generation process:

```python
idx = torch.zeros((1, 1), dtype=torch.long)
generated_ids = m.generate(idx, max_new_tokens=100)[0].tolist()
print(decode(generated_ids))
```

The model should generally produce a lower loss and more recognizable character patterns after training. A bigram model still has limited ability to learn longer-range language structure.

## 11. Complete Workflow Summary

1. **Initialize:** Create the model with randomly initialized weights.
2. **Predict:** Generate logits for the next character.
3. **Calculate loss:** Compare predictions with correct targets.
4. **Backpropagate:** Calculate gradients using `loss.backward()`.
5. **Update weights:** Use `optimizer.step()`.
6. **Repeat training:** Improve predictions across many batches.
7. **Generate:** Sample next characters using softmax and `torch.multinomial()`.
8. **Decode:** Convert token IDs into readable text.

**Key distinction:** `forward()` makes predictions, `loss.backward()` calculates gradients, and `optimizer.step()` updates the model. These are separate parts of the learning process.

### ----------------------------------------------------------------------------------------------------------------------------
#### as the ouput of this , is very ranodom 
### so , lets train the model so output become less random 

## Training 

In [14]:
# Create a PyTorch Optimizer 
optimizer = torch.optim.AdamW(m.parameters(),lr = 1e-3)

In [17]:
# Creating optimizer object , which will basicaly takes the gradient and update parameters using gradients 

batch_size = 32 # increased from 4 to 32 
for steps in range(100):

    #sample a batch of  data 
    xb, yb = get_batch('train')

    # evaluate the loss 
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

4.352577209472656


In [20]:
idx = torch.zeros((1, 1),dtype = torch.long)
generated_ids = m.generate(idx, max_new_tokens=1000)[0].tolist()
print(decode(generated_ids))


itJTn
FXzUpWyOWYH-oOrpWxPDFctWd!DWpGoLkQN?nnqupOL!K!Pr!VsvFhwPJ XiZcimzRCDh- 3N, ERJ!KO$faVl?ZDAWKRFcd$EsuoPGpBib
AEqKUxJ.-.-SolKbKPUBJuAimu;jiIYoFiLxm
KWaGMuaw!x.NFTYLZIYXdvG;ZhwBpB?ti zrwPQDT'b?Fcog?Hfkvfac?C3hJXKTudipZKSISoc?ixJOPBeKQvbJ!$pWXQYKupW33miGg O,c,iSByIEmdooGTAmgGdM&Dc?vmC;LvYT?OHfwl!zeFfOyeDZFV
pWC:NkzJXdg.wl,JvCgz&UHGh!?yQYW
AE..bz$AUpjwthWtLDNUSHorkFf.QEYdi3X,lQB.qL.uV&zsvvkF
mCDUvnGevy!oZgMcMjj?XyQRh'LO3XVkRJFq?FF?PaH'h
gzcor;dldE&zZ!,T'.jvkbBjjjvE.BqcX3n,iTjETFOWfiS'.s;X!oyEZcdZFV&FTdLo$ctqcIpWciwPmtbDi;$MHRUmJk.nu:iUXZcYWc? k:xusu.mx:X.EJUp;!yEkIY;jLCEMs;kRQEqvQPGaudRr,evhAEs-YLzO!hq?w-D-YW-- YZFf,mx,PD
F
BW
i:nTQ!K&GjE&qQZ?nHiVT?Q!zh!rGDZcfMJ?VT3xWbJOynepWKQ!waaW

:feth:EobnPDT g.-avYjEsf
MDAwmxtwg3N3?gzUAaGMqu.:rp:JPCLpqcxtE?3ZhJ.qh:T-U
FO,T?3keInB,Fg;DNZdemm;g?h!? nLJ?yE.-ScIEtwmo-YXKCDRpUzc'cZOjhwgek$CKLZh,fgjjvjWXD-$AnW$ol.- OCDNTfkiwwcdZfSXi3VVGa
:uXT
ZtELFVoc-nGFm
:JSndHh!OHJb:CthJ!:xOwNdgNMI.!GX
HiwmtqhJnMZd3DUxIE$rMPUz33Q!YWnwp,m-z:yZ.GorpQ!rjyJOzRcLe
:GEs

##  Moving to a script (bigram.py) with practical upgrades

In [24]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Hyperparameters ---
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2

# To this for a Mac:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
eval_iters = 200
# -----------------------------------

torch.manual_seed(19)

# --- Data Loading & Tokenization ---
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# Data loading function
def get_batch(split):
    # Generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# --- Bigram Language Model Definition ---
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (Batch, Time, Channels/Vocab Size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens (not strictly needed for bigram, but good practice for later)
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)

# --- Optimizer Setup & Training Loop ---
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# --- Generate from the Model ---
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.6346, val loss 4.6499
step 300: train loss 2.7888, val loss 2.8111
step 600: train loss 2.5532, val loss 2.5667
step 900: train loss 2.4977, val loss 2.5223
step 1200: train loss 2.4843, val loss 2.5046
step 1500: train loss 2.4574, val loss 2.5038
step 1800: train loss 2.4647, val loss 2.4910
step 2100: train loss 2.4659, val loss 2.4959
step 2400: train loss 2.4601, val loss 2.4887
step 2700: train loss 2.4672, val loss 2.4891

NISepith rar my t hith ouse, he oud.
Th clord ssu LLur slourethat angre bus.

As be s, me toros ive my ton
Yorusenode?

S:

I mio ed we Low--hod
Fis, ou hankell, red, ioug mensininorit E:


A m ds? whence
Yonto o ndelolyevewan d.
Peame yo heapellon m mand be ggtheanthend, nde?
TousHeantith, Leaiang T: LISire grve cool IChes youse tchind houl:
WAnd diun:
MENaroy.
Cou fo s ar s
ONGoxYor.
aicakillloumuctlllithacaton aindouro fft,
I, Buemyord, ly I hofa toulley ant, anght me?

WAnousiu! thyordouruth


## The Mathematical trick in self-attention 

In [ ]:
## Consider the following toy example : 

torch.manual_seed(19)
B,T,C = 4,8,2 # batch , time , channels
# at this time these '8' tokens are not talking to each other . 
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [28]:
# consider the following toy example:

torch.manual_seed(19)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C)) # (Note: in the actual video code it is initialized as torch.zeros_like(x))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)


In [29]:
x[0]

tensor([[ 1.5507,  0.6488],
        [ 0.0273, -0.4079],
        [ 0.6885,  0.4269],
        [-0.9449,  0.1573],
        [-2.1254, -0.1581],
        [ 2.0588,  1.4157],
        [ 0.1141,  0.2676],
        [ 0.0713,  0.0943]])

In [30]:
xbow[0]

tensor([[ 1.5507,  0.6488],
        [ 0.7890,  0.1205],
        [ 0.7555,  0.2226],
        [ 0.3304,  0.2063],
        [-0.1608,  0.1334],
        [ 0.2092,  0.3471],
        [ 0.1956,  0.3358],
        [ 0.1800,  0.3056]])

* as we can see that at first rows both x[0] and xbow[0] are equal but from 2nd rows both are entirely different 


* as becoz , xbow[] calculates avg. of the past , for example 4th rows in xbow is an average of last three rows of x[]

#### --> But, doing like this is very inefficient 

### so , the tricks to make it efficient , is by using matrix multiplication 

In [ ]:
torch.tril(torch.ones(3,3)) # torch.tril to generate a lower traingle 

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [33]:
torch.manual_seed(19)
a = torch.tril(torch.ones(3,3))
b = torch.randint(0,10,(3,2)).float()
c = a @ b 
print('a = ')
print(a)
print('-----------------------------')
print('b = ')
print(b)
print('------------------------------')
print('c = ')
print(c)

a = 
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
-----------------------------
b = 
tensor([[5., 8.],
        [5., 0.],
        [4., 9.]])
------------------------------
c = 
tensor([[ 5.,  8.],
        [10.,  8.],
        [14., 17.]])
